In [1]:
import pandas as pd
import warnings
import sys
import os
import pickle
import numpy as np

sys.path.append(os.path.abspath(".."))

from src.app.core.api import Features
from src.app.core.model import Model

warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

In [2]:
DATA_PATH = "../hw6/data/val_features.csv"

MODEL_PATH = "../hw6/models/logistic_regression.pkl"

SCALER_PATH = "../hw6/models/logistic_regression_scaler.pkl"

IDX_TO_COMPARE_COUNT = 1000

In [3]:
df_val = pd.read_csv(DATA_PATH)
print(f"Валидационная выборка: {df_val.shape}")

Валидационная выборка: (52656, 10)


Логистическая регрессия была обучена на масштабированных признаках

In [4]:
with open(SCALER_PATH, "rb") as f:
    scaler = pickle.load(f)

df_scaled = scaler.transform(df_val)

Обертка

In [5]:
wrapper = Model(MODEL_PATH, SCALER_PATH)

Модель

In [6]:
with open(MODEL_PATH, "rb") as f:
    model = pickle.load(f)

Проверяем пробы на обертке и модели

In [7]:
probas_wrapper = wrapper.predict_proba_batch(df_val[:IDX_TO_COMPARE_COUNT])

In [8]:
probas_model = model.predict_proba(df_scaled[:IDX_TO_COMPARE_COUNT])[:, 1]

In [9]:
if all(probas_wrapper == probas_model):
    print("Все вероятности совпадают")
else:
    print("Вероятности не совпадают")

Все вероятности совпадают


Проверка калькулятора

In [10]:
df_val["features"] = df_val.apply(dict, axis=1)

df_val["scoring_results"] = df_val["features"].apply(lambda x: wrapper.get_scoring_result(Features(**x)))

df_val

,weighted_ext_score,ext_source_3,ext_source_2,days_registration,days_birth,days_id_publish,annuity_to_income_proportion,interest_rate,days_employed,amt_annuity,features,scoring_results
0,0.336558,0.510089,0.468826,-13252.0,-24248,-4638,0.130267,10.286383,365243,17586.0,"{'weighted_ext_score': 0.3365582763192264, 'ex...",ScoringResult(decision=<ScoringDecision.DECLIN...
1,0.656764,0.812823,0.585148,-2832.0,-13436,-5901,0.084543,5.262763,-193,13315.5,"{'weighted_ext_score': 0.6567637677860341, 'ex...",ScoringResult(decision=<ScoringDecision.ACCEPT...
2,0.660741,0.665855,0.415780,-9257.0,-20066,-3474,0.185457,16.031722,-153,29209.5,"{'weighted_ext_score': 0.660740637551938, 'ext...",ScoringResult(decision=<ScoringDecision.ACCEPT...
3,0.225514,0.535276,0.266520,-4957.0,-16953,-510,0.209240,0.926304,-4400,23539.5,"{'weighted_ext_score': 0.2255135445531536, 'ex...",ScoringResult(decision=<ScoringDecision.DECLIN...
4,0.503131,0.792264,0.667729,-3208.0,-9028,-1695,0.165450,2.384518,-1056,44671.5,"{'weighted_ext_score': 0.503130800946699, 'ext...",ScoringResult(decision=<ScoringDecision.ACCEPT...
...,...,...,...,...,...,...,...,...,...,...,...,...
52651,0.240077,0.174564,0.545486,-1604.0,-16814,-364,0.288800,6.481854,-2507,45486.0,"{'weighted_ext_score': 0.2400769733885292, 'ex...",ScoringResult(decision=<ScoringDecision.DECLIN...
52652,0.370412,0.718033,0.341344,-7200.0,-19761,-3286,0.117533,1.553823,-528,23800.5,"{'weighted_ext_score': 0.3704119379973846, 'ex...",ScoringResult(decision=<ScoringDecision.DECLIN...
52653,0.577943,0.646330,0.728201,-5353.0,-11355,-3912,0.183067,17.731090,-1189,24714.0,"{'weighted_ext_score': 0.5779430730754505, 'ex...",ScoringResult(decision=<ScoringDecision.ACCEPT...
52654,0.624749,0.372334,0.709926,-989.0,-16515,-58,0.147111,1.619520,-2871,47664.0,"{'weighted_ext_score': 0.6247487539166623, 'ex...",ScoringResult(decision=<ScoringDecision.DECLIN...


In [11]:
idx = 18

result = df_val.loc[idx, "scoring_results"]

print("Информация о клиенте")
print(df_val.iloc[idx])

print("Результат")
print(f"Решение о выдаче займа: {result.decision}")
print(f"Сумма: {result.amount}")
print(f"threshold: {result.threshold}")
print(f"proba: {result.proba:.8f}")
print(f"proba модели напрямую: {model.predict_proba(df_scaled[idx : idx + 1])[:, 1]}")

Информация о клиенте
weighted_ext_score                                                       0.437571
ext_source_3                                                             0.698667
ext_source_2                                                             0.569975
days_registration                                                         -3377.0
days_birth                                                                 -21040
days_id_publish                                                             -4385
annuity_to_income_proportion                                             0.333333
interest_rate                                                           21.314007
days_employed                                                               -2228
amt_annuity                                                              225000.0
features                        {'weighted_ext_score': 0.4375707631035926, 'ex...
scoring_results                 ScoringResult(decision=<ScoringDecision.ACCEP

Вывод для n заявлений

In [12]:
n = 1000

df_subset = df_val.head(n).copy()

results = []
for idx in range(len(df_subset)):
    result = df_subset.iloc[idx]["scoring_results"]
    row = df_subset.iloc[idx]
    
    results.append(
        {
            "proba": result.proba,
            "decision": result.decision.name,
            "amount": result.amount,
            "weighted_ext_score": row["weighted_ext_score"],
            "ext_source_3": row["ext_source_3"],
            "ext_source_2": row["ext_source_2"],
            "days_registration": row["days_registration"],
            "days_birth": row["days_birth"],
            "days_id_publish": row["days_id_publish"],
            "annuity_to_income_proportion": row["annuity_to_income_proportion"],
            "interest_rate": row["interest_rate"],
            "days_employed": row["days_employed"],
            "amt_annuity": row["amt_annuity"],
        }
    )

results = pd.DataFrame(results)

results.to_csv(f"../{n}_заявлений.csv")

results

,proba,decision,amount,weighted_ext_score,ext_source_3,ext_source_2,days_registration,days_birth,days_id_publish,annuity_to_income_proportion,interest_rate,days_employed,amt_annuity
0,0.332764,DECLINED,0,0.336558,0.510089,0.468826,-13252.0,-24248,-4638,0.130267,10.286383,365243,17586.0
1,0.164767,ACCEPTED,168000,0.656764,0.812823,0.585148,-2832.0,-13436,-5901,0.084543,5.262763,-193,13315.5
2,0.209276,ACCEPTED,36800,0.660741,0.665855,0.415780,-9257.0,-20066,-3474,0.185457,16.031722,-153,29209.5
3,0.666126,DECLINED,0,0.225514,0.535276,0.266520,-4957.0,-16953,-510,0.209240,0.926304,-4400,23539.5
4,0.251549,ACCEPTED,35000,0.503131,0.792264,0.667729,-3208.0,-9028,-1695,0.165450,2.384518,-1056,44671.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0.420819,DECLINED,0,0.304658,0.385915,0.508934,-6142.0,-20668,-4173,0.055556,10.620976,365243,6750.0
996,0.160419,ACCEPTED,147000,0.501771,0.768808,0.689692,-149.0,-22652,-3972,0.182171,4.859400,365243,28692.0
997,0.607638,DECLINED,0,0.303725,0.331251,0.567003,-2429.0,-14499,-2447,0.292133,5.308712,-2482,39438.0
998,0.519435,DECLINED,0,0.444344,0.408359,0.458267,-4290.0,-10396,-3057,0.179771,6.537401,-204,38830.5
